# 01 — Data Cleaning & Exploratory Data Analysis

**Day 2 objective:** Load all Olist CSV files, perform a thorough EDA (shapes, dtypes,
null values, distributions, duplicates, date ranges), clean each dataset, merge into
one master analytical DataFrame, and save to `data/processed/`.

---

### Notebooks in this project
| # | Notebook | Topic |
|---|----------|-------|
| 01 | `01_data_cleaning.ipynb` | ← **you are here** |
| 02 | `02_revenue_analysis.ipynb` | Revenue & payment trends |
| 03 | `03_customer_segmentation.ipynb` | RFM segmentation |
| 04 | `04_product_analysis.ipynb` | Product category deep-dive |
| 05 | `05_geographic_analysis.ipynb` | State-level geographic analysis |
| 06 | `06_profitability_dashboard.ipynb` | Profitability model |

---

## 0. Setup & Imports

In [ ]:
import sys
import warnings
warnings.filterwarnings('ignore')

sys.path.append('../src')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

from data_cleaning import (
    load_raw_datasets,
    inspect_dataset,
    missing_value_report,
    clean_orders,
    clean_payments,
    clean_products,
    clean_customers,
    build_master_df,
    save_processed,
)

# Consistent plot style
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor':   'white',
    'axes.spines.top':  False,
    'axes.spines.right': False,
})

print('Libraries loaded ✅')

---
## 1. Load Raw Datasets

All 8 Olist CSV files are loaded from `data/raw/`.

In [ ]:
datasets = load_raw_datasets()

In [ ]:
# Quick size overview
print(f"{'Dataset':<30} {'Rows':>8} {'Cols':>6}")
print('-' * 46)
for name, df in datasets.items():
    print(f"{name:<30} {df.shape[0]:>8,} {df.shape[1]:>6}")

---
## 2. Schema Inspection

For each dataset: column names, dtypes, null counts, and a sample.

In [ ]:
for name, df in datasets.items():
    inspect_dataset(df, name)

---
## 3. Missing Value Analysis

In [ ]:
missing_value_report(datasets)

In [ ]:
# Visualise missing values per dataset
fig, axes = plt.subplots(2, 4, figsize=(22, 9))
axes = axes.flatten()

for i, (name, df) in enumerate(datasets.items()):
    null_pct = (df.isnull().mean() * 100).sort_values(ascending=False)
    null_pct = null_pct[null_pct > 0]
    if not null_pct.empty:
        axes[i].barh(null_pct.index, null_pct.values, color='#EF4444', alpha=0.85)
        for j, v in enumerate(null_pct.values):
            axes[i].text(v + 0.3, j, f'{v:.1f}%', va='center', fontsize=8)
        axes[i].set_xlabel('% Missing')
    else:
        axes[i].text(0.5, 0.5, 'No nulls ✅', ha='center', va='center', fontsize=13)
        axes[i].set_xticks([])
        axes[i].set_yticks([])
    axes[i].set_title(name, fontsize=10, fontweight='bold')

plt.suptitle('Missing Values by Dataset', fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('../images/missing_values.png', dpi=150, bbox_inches='tight')
plt.show()
print('📊 Chart saved → images/missing_values.png')

---
## 4. Duplicate Check

In [ ]:
print(f"{'Dataset':<30} {'Duplicate Rows':>16}")
print('-' * 48)
for name, df in datasets.items():
    dupes = df.duplicated().sum()
    flag = '⚠️' if dupes > 0 else '✅'
    print(f"{name:<30} {dupes:>12,}  {flag}")

---
## 5. Orders Dataset — Deep Dive

### 5a. Order Status Distribution

In [ ]:
orders_raw = datasets['orders'].copy()

status_counts = orders_raw['order_status'].value_counts()
status_pct    = (status_counts / len(orders_raw) * 100).round(2)

print('Order Status Breakdown:')
print('=' * 38)
for status, cnt in status_counts.items():
    print(f"  {status:<25} {cnt:>6,}  ({status_pct[status]:.1f}%)")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
colors = ['#2563EB' if s == 'delivered' else '#D1D5DB' for s in status_counts.index]
bars = ax.bar(status_counts.index, status_counts.values, color=colors, alpha=0.9)
ax.bar_label(bars, labels=[f'{v:,}' for v in status_counts.values], padding=4, fontsize=9)
ax.set_title('Order Status Distribution', fontsize=14, fontweight='bold')
ax.set_xlabel('Status')
ax.set_ylabel('Number of Orders')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.savefig('../images/order_status_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

### 5b. Date Range & Order Volume Over Time

In [ ]:
orders_raw['order_purchase_timestamp'] = pd.to_datetime(
    orders_raw['order_purchase_timestamp'], errors='coerce'
)

print(f"Earliest order : {orders_raw['order_purchase_timestamp'].min()}")
print(f"Latest order   : {orders_raw['order_purchase_timestamp'].max()}")
print(f"Date span      : {(orders_raw['order_purchase_timestamp'].max() - orders_raw['order_purchase_timestamp'].min()).days} days")

In [ ]:
# Monthly order volume (all statuses)
orders_raw['yearmonth'] = orders_raw['order_purchase_timestamp'].dt.to_period('M')
monthly_orders = orders_raw.groupby('yearmonth')['order_id'].count().reset_index()
monthly_orders['yearmonth'] = monthly_orders['yearmonth'].astype(str)

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(monthly_orders['yearmonth'], monthly_orders['order_id'],
        marker='o', linewidth=2.5, color='#2563EB', markersize=4)
ax.fill_between(monthly_orders['yearmonth'], monthly_orders['order_id'],
                alpha=0.12, color='#2563EB')
ax.set_title('Monthly Order Volume (All Statuses)', fontsize=14, fontweight='bold')
ax.set_xlabel('Month')
ax.set_ylabel('Number of Orders')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('../images/monthly_order_volume.png', dpi=150, bbox_inches='tight')
plt.show()

### 5c. Delivery Time Distribution

In [ ]:
delivered = orders_raw[orders_raw['order_status'] == 'delivered'].copy()
delivered['order_delivered_customer_date'] = pd.to_datetime(
    delivered['order_delivered_customer_date'], errors='coerce'
)
delivered['delivery_days'] = (
    delivered['order_delivered_customer_date'] - delivered['order_purchase_timestamp']
).dt.days
delivered = delivered.dropna(subset=['delivery_days'])
delivered = delivered[delivered['delivery_days'] > 0]

print(f"Delivery time stats (days):")
print(delivered['delivery_days'].describe().round(1).to_string())

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
ax.hist(delivered['delivery_days'].clip(upper=60), bins=60,
        color='#2563EB', alpha=0.8, edgecolor='white')
ax.axvline(delivered['delivery_days'].median(), color='#EF4444',
           linestyle='--', linewidth=2, label=f"Median: {delivered['delivery_days'].median():.0f} days")
ax.axvline(delivered['delivery_days'].mean(), color='#F59E0B',
           linestyle='--', linewidth=2, label=f"Mean: {delivered['delivery_days'].mean():.1f} days")
ax.set_title('Delivery Time Distribution', fontsize=14, fontweight='bold')
ax.set_xlabel('Days from Purchase to Delivery')
ax.set_ylabel('Number of Orders')
ax.legend()
plt.tight_layout()
plt.savefig('../images/delivery_time_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 6. Payments Dataset — Deep Dive

In [ ]:
payments_raw = datasets['payments'].copy()

print('Payment Type Counts:')
print(payments_raw['payment_type'].value_counts())
print(f"\nPayment value range: R${payments_raw['payment_value'].min():.2f} — R${payments_raw['payment_value'].max():,.2f}")
print(f"Mean payment value : R${payments_raw['payment_value'].mean():,.2f}")
print(f"Zero-value rows    : {(payments_raw['payment_value'] == 0).sum()}")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Payment type share
pay_counts = payments_raw['payment_type'].value_counts()
ax1.pie(pay_counts.values, labels=pay_counts.index, autopct='%1.1f%%',
        startangle=140, colors=sns.color_palette('muted', len(pay_counts)))
ax1.set_title('Payment Type Share', fontweight='bold')

# Payment value distribution (capped for readability)
cap = payments_raw['payment_value'].quantile(0.99)
ax2.hist(payments_raw['payment_value'].clip(upper=cap), bins=60,
         color='#10B981', alpha=0.8, edgecolor='white')
ax2.set_title('Payment Value Distribution (99th pct cap)', fontweight='bold')
ax2.set_xlabel('Payment Value (R$)')
ax2.set_ylabel('Count')
ax2.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'R${x:,.0f}'))

plt.tight_layout()
plt.savefig('../images/payment_overview.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 7. Products Dataset — Deep Dive

In [ ]:
products_raw = datasets['products'].copy()

print(f"Total unique products  : {products_raw['product_id'].nunique():,}")
print(f"Total categories       : {products_raw['product_category_name'].nunique()}")
print(f"Missing category names : {products_raw['product_category_name'].isnull().sum()}")
print()
print('Top 10 categories by product count:')
print(products_raw['product_category_name'].value_counts().head(10).to_string())

In [ ]:
# Physical dimensions overview
dim_cols = ['product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm']
available_dims = [c for c in dim_cols if c in products_raw.columns]

if available_dims:
    fig, axes = plt.subplots(1, len(available_dims), figsize=(16, 4))
    for ax, col in zip(axes, available_dims):
        cap = products_raw[col].quantile(0.99)
        ax.hist(products_raw[col].dropna().clip(upper=cap), bins=50,
                color='#F59E0B', alpha=0.8, edgecolor='white')
        ax.set_title(col.replace('product_', '').replace('_', ' ').title(), fontweight='bold')
        ax.set_xlabel('Value')
    plt.suptitle('Product Physical Dimensions (99th pct cap)', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig('../images/product_dimensions.png', dpi=150, bbox_inches='tight')
    plt.show()

---
## 8. Customers Dataset — Deep Dive

In [ ]:
customers_raw = datasets['customers'].copy()

print(f"Total customer records    : {len(customers_raw):,}")
print(f"Unique customer IDs       : {customers_raw['customer_id'].nunique():,}")
print(f"Unique customer_unique_id : {customers_raw['customer_unique_id'].nunique():,}")
print(f"Unique states             : {customers_raw['customer_state'].nunique()}")
print()
print('Top 10 states by customer count:')
print(customers_raw['customer_state'].value_counts().head(10).to_string())

In [ ]:
top_states = customers_raw['customer_state'].value_counts().head(15)

fig, ax = plt.subplots(figsize=(12, 5))
sns.barplot(x=top_states.index, y=top_states.values, palette='Blues_d', ax=ax)
ax.set_title('Customers by State (Top 15)', fontsize=14, fontweight='bold')
ax.set_xlabel('State')
ax.set_ylabel('Customer Count')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
plt.tight_layout()
plt.savefig('../images/customers_by_state.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 9. Order Items Dataset — Deep Dive

In [ ]:
items_raw = datasets['order_items'].copy()

print(f"Total order item rows  : {len(items_raw):,}")
print(f"Unique orders          : {items_raw['order_id'].nunique():,}")
print(f"Unique products        : {items_raw['product_id'].nunique():,}")
print(f"Unique sellers         : {items_raw['seller_id'].nunique():,}")
print()
print('Price stats (R$):')
print(items_raw['price'].describe().round(2).to_string())
print()
print('Freight value stats (R$):')
print(items_raw['freight_value'].describe().round(2).to_string())

In [ ]:
# Items per order distribution
items_per_order = items_raw.groupby('order_id')['order_item_id'].max()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Price distribution
cap_price = items_raw['price'].quantile(0.99)
ax1.hist(items_raw['price'].clip(upper=cap_price), bins=60,
         color='#2563EB', alpha=0.8, edgecolor='white')
ax1.set_title('Item Price Distribution (99th pct cap)', fontweight='bold')
ax1.set_xlabel('Price (R$)')
ax1.set_ylabel('Count')

# Items per order
ax2.hist(items_per_order.clip(upper=10), bins=range(1, 12),
         color='#10B981', alpha=0.8, edgecolor='white', rwidth=0.85)
ax2.set_title('Items per Order Distribution', fontweight='bold')
ax2.set_xlabel('Number of Items')
ax2.set_ylabel('Number of Orders')
ax2.set_xticks(range(1, 11))

plt.tight_layout()
plt.savefig('../images/order_items_overview.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 10. Build & Validate Master DataFrame

In [ ]:
master = build_master_df(datasets)

In [ ]:
print('Master DataFrame — shape:', master.shape)
print('\nColumns:')
for col in master.columns:
    print(f"  {col:<40} {str(master[col].dtype):<12} nulls: {master[col].isnull().sum():,}")

In [ ]:
print('Date range in master:')
print(f"  First order : {master['order_purchase_timestamp'].min()}")
print(f"  Last order  : {master['order_purchase_timestamp'].max()}")
print()
print('Quick sanity checks:')
print(f"  Unique orders    : {master['order_id'].nunique():,}")
print(f"  Unique customers : {master['customer_unique_id'].nunique():,}")
print(f"  Unique products  : {master['product_id'].nunique():,}")
print(f"  Total revenue    : R${master['payment_value'].sum():,.2f}")
print(f"  Avg order value  : R${master['payment_value'].mean():,.2f}")

In [ ]:
master.head()

---
## 11. Save Processed Data

In [ ]:
save_processed(master, 'master.csv')
print()
print('✅ Day 2 complete — data cleaned and master DataFrame saved.')
print('➡️  Next: 02_revenue_analysis.ipynb')